# TourismGPT

# TourismGPT: Industry-Specific Large Language Model for the Tourism Industry

## Project Summary

TourismGPT is an industry-specific conversational AI system developed to provide accurate, context-aware responses to tourism and travel-related queries. Unlike general-purpose chatbots, TourismGPT combines a fine-tuned Large Language Model (LLM) with Retrieval-Augmented Generation (RAG) to generate reliable answers using domain-specific knowledge.

The project uses Microsoft's Phi-3 Mini pre-trained language model and fine-tunes it using QLoRA (Quantized Low-Rank Adaptation) on a curated tourism dataset. To improve factual accuracy and reduce hallucinations, the system incorporates a FAISS-based retrieval mechanism that searches tourism documents before generating responses.

This project demonstrates how pre-trained LLMs can be adapted to solve real-world industry problems by combining transfer learning, efficient fine-tuning techniques, and document retrieval into a practical conversational assistant.

---

## Project Objectives

- Build an industry-specific chatbot for the Tourism and Hospitality sector.
- Fine-tune a pre-trained Phi-3 Mini model using tourism-specific instruction-response data.
- Improve response accuracy using Retrieval-Augmented Generation (RAG).
- Develop an interactive chatbot capable of answering travel-related questions.
- Demonstrate the practical application of domain-specific Large Language Models.

# Part 1 – Fine-Tuning Microsoft Phi-3 Mini Using QLoRA

## Business Problem

General-purpose language models possess broad knowledge but often struggle with industry-specific terminology, travel recommendations, destination details, and contextual tourism information. Their responses may also become outdated or hallucinated when answering specialized questions.

To address these challenges, this project develops TourismGPT, an AI assistant specifically designed for the tourism industry. The model is fine-tuned using tourism-specific instruction-response pairs and enhanced with Retrieval-Augmented Generation (RAG), allowing it to retrieve relevant documents before generating responses.

## Technology Stack

- Python
- Google Colab (T4 GPU)
- Hugging Face Transformers
- Microsoft Phi-3 Mini
- Unsloth
- PEFT (QLoRA)
- FAISS Vector Database
- Sentence Transformers
- Gradio
- JSONL Dataset

## Project Workflow

1. Collect tourism-specific data from reliable public sources.
2. Clean and preprocess the dataset.
3. Convert the data into instruction-response format.
4. Fine-tune the Phi-3 Mini model using QLoRA.
5. Build a FAISS-based retrieval system.
6. Integrate retrieval with the fine-tuned LLM.
7. Deploy an interactive chatbot using Gradio.
8. Evaluate the chatbot using tourism-related queries.

## Environment Setup

Before training the model, all required libraries are installed. These libraries provide support for loading pre-trained language models, parameter-efficient fine-tuning, dataset processing, vector search, and model deployment.

### Libraries Used

- **Transformers** – Loads and manages the pre-trained Phi-3 model.
- **Datasets** – Processes the tourism instruction dataset.
- **PEFT (Parameter-Efficient Fine-Tuning)** – Implements LoRA and QLoRA for efficient model adaptation.
- **Unsloth** – Accelerates fine-tuning while reducing GPU memory consumption.
- **BitsAndBytes** – Enables 4-bit quantization for memory-efficient training.
- **Sentence Transformers** – Generates dense vector embeddings for tourism documents.
- **FAISS** – Performs fast similarity search during Retrieval-Augmented Generation (RAG).
- **Gradio** – Creates an interactive web interface for the chatbot.

Installing these dependencies ensures that the complete TourismGPT pipeline—from fine-tuning to deployment—runs successfully within the Google Colab environment.

In [1]:
# CELL 1 — Install dependencies (restart runtime if prompted)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets huggingface_hub

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-szob5b_t/unsloth_21e913b2c27743519fb53b2d81516f19
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-szob5b_t/unsloth_21e913b2c27743519fb53b2d81516f19
  Resolved https://github.com/unslothai/unsloth.git to commit 278e9e7921a56c603a3384e1bdc8562c4e354858
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 24.0 MB/s eta 0:00:00
   

In [2]:
# CELL 2 — Mount Google Drive and copy dataset
from google.colab import drive
drive.mount('/content/drive')

# Option A: copy from Drive (if you uploaded tourism_finetune.jsonl to Drive)
# !cp "/content/drive/MyDrive/tourism_finetune.jsonl" /content/tourism_finetune.jsonl

# Option B: upload directly via Colab Files panel (left sidebar → upload icon)
# then set JSONL_PATH = "/content/tourism_finetune.jsonl" in Cell 3

Mounted at /content/drive


## Import Required Libraries

After installing the required packages, the necessary Python libraries are imported. These libraries support model loading, dataset handling, training, evaluation, retrieval, and deployment.

Each library contributes to a different stage of the pipeline, enabling the development of an end-to-end industry-specific conversational AI system.

In [3]:
# CELL 3 — Imports and config
import json
import os
import torch
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel

# ── Paths ──────────────────────────────────────────────────────────────────
JSONL_PATH   = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/datasets/tourism_finetune.jsonl"
OUTPUT_DIR   = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter"
DRIVE_SAVE   = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter"

# ── Model config ───────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Phi-3-mini-4k-instruct"
MAX_SEQ_LEN  = 2048
DTYPE        = None
LOAD_IN_4BIT = True

# ── LoRA config ────────────────────────────────────────────────────────────
LORA_RANK    = 16
LORA_ALPHA   = 16
LORA_DROPOUT = 0

# ── Training config ────────────────────────────────────────────────────────
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
WARMUP_STEPS = 5
MAX_STEPS    = 300
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
WEIGHT_DECAY = 0.01
SAVE_STEPS   = 100
LOGGING_STEPS = 25
FP16         = not torch.cuda.is_bf16_supported()
BF16         = torch.cuda.is_bf16_supported()
SEED         = 42

print("Config ready ✓")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Config ready ✓


## Why Microsoft Phi-3 Mini?

The Microsoft Phi-3 Mini model was selected because it provides an excellent balance between performance, efficiency, and computational requirements. Despite its relatively small size compared to larger language models, Phi-3 Mini demonstrates strong reasoning capabilities and performs well on instruction-following tasks.

The model is lightweight enough to be fine-tuned on a Google Colab T4 GPU while still producing high-quality responses. This makes it an ideal choice for developing an industry-specific conversational assistant within limited computational resources.

In [4]:
# CELL 4 — Load Phi-3 Mini with Unsloth (4-bit QLoRA)
print("Loading Phi-3 Mini ...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    use_rslora                 = False,
    loftq_config               = None,
)

print(model.print_trainable_parameters())

Loading Phi-3 Mini ...
==((====))==  Unsloth 2026.7.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 29,884,416 || all params: 3,850,963,968 || trainable%: 0.7760
None


## Load the Tourism Dataset

The chatbot is trained using a tourism-specific instruction dataset containing question-answer pairs related to travel destinations, attractions, transportation, accommodation, local culture, and travel planning.

Using a domain-specific dataset enables the language model to understand tourism terminology and generate responses that are more accurate and contextually relevant than those produced by a general-purpose chatbot.

In [5]:
# CELL 5 — Load and format dataset
EOS = tokenizer.eos_token

def format_example(row):
    instruction = row["instruction"].strip()
    output      = row["output"].strip()
    return (
        f"<|user|>\n{instruction}<|end|>\n"
        f"<|assistant|>\n{output}<|end|>\n"
        f"{EOS}"
    )

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f"Loaded {len(records)} examples")
    texts = [format_example(r) for r in records]
    return Dataset.from_dict({"text": texts})

dataset = load_jsonl(JSONL_PATH)

print("\n── Sample training text ──────────────────────────────")
print(dataset["text"][0][:400])
print("──────────────────────────────────────────────────────")

Loaded 2220 examples

── Sample training text ──────────────────────────────
<|user|>
where can I check if there are any news on the rebate?<|end|>
<|assistant|>
I've observed that you're eager to stay updated on any news about your rebate. To check the latest updates, you can visit our website's "My Account" section. Log in using your credentials, and navigate to the "Rebate Status" or "Refund Status" page. This page will provide you with real-time information on the prog
──────────────────────────────────────────────────────


## Tokenization

Before training, the tokenizer converts human-readable text into numerical token IDs that the language model can understand.

Both user instructions and expected responses are tokenized using the tokenizer associated with the Phi-3 Mini model. This ensures consistency between pre-training and fine-tuning while preserving the model's original vocabulary and language understanding.

In [6]:
# CELL 6x — Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    args               = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        dataset_num_proc            = 2,
        packing                     = False,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        max_steps                   = MAX_STEPS,
        learning_rate               = LEARNING_RATE,
        fp16                        = FP16,
        bf16                        = BF16,
        logging_steps               = LOGGING_STEPS,
        optim                       = "adamw_8bit",
        weight_decay                = WEIGHT_DECAY,
        lr_scheduler_type           = LR_SCHEDULER,
        seed                        = SEED,
        output_dir                  = OUTPUT_DIR,
        save_steps                  = SAVE_STEPS,
        save_total_limit            = 2,
        report_to                   = "none",
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU: {gpu_stats.name}  |  {max_memory} GB total")
print("\nStarting training ...")

trainer_stats = trainer.train()

print("\n── Training complete ──")
print(f"  Runtime  : {trainer_stats.metrics['train_runtime']:.0f} s")
print(f"  Samples/s: {trainer_stats.metrics['train_samples_per_second']:.1f}")
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"  Peak VRAM: {used_memory} GB / {max_memory} GB")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2220 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
GPU: Tesla T4  |  14.563 GB total

Starting training ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,220 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
25,1.723475
50,1.018992
75,0.761257
100,0.661865
125,0.683369
150,0.683062
175,0.555806
200,0.624294
225,0.634532
250,0.605896


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter/checkpoint-100/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter/checkpoint-100.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter/checkpoint-300/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter/checkpoint-300.



── Training complete ──
  Runtime  : 1014 s
  Samples/s: 2.4
  Peak VRAM: 2.816 GB / 14.563 GB


In [8]:
# CELL 7 — Save LoRA adapter
print(f"Saving adapter to {OUTPUT_DIR} ...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Adapter saved ✓")

# Copy to Google Drive so it survives after the session ends
print(f"Adapter is already saved in Google Drive: {OUTPUT_DIR} ✓")

Saving adapter to /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter ...
Adapter saved ✓
Adapter is already saved in Google Drive: /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter ✓


In [9]:
# CELL 8 — Quick inference test
def ask(question, max_new_tokens=300):
    FastLanguageModel.for_inference(model)
    prompt = f"<|user|>\n{question}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_questions = [
    "Plan a 5-day itinerary for Tokyo for a solo budget traveller.",
    "Compare Bali vs Thailand for a honeymoon trip.",
    "What is the estimated budget for a week in Paris?",
    "How do I book a hotel with free cancellation on Booking.com?",
    "My flight was cancelled — how do I claim a refund?",
    "What cultural customs should I know before visiting Japan?",
]

print("── Inference tests ───────────────────────────────────")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)}")
    print("─" * 60)

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


── Inference tests ───────────────────────────────────

Q: Plan a 5-day itinerary for Tokyo for a solo budget traveller.


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

A: Here is a 5-day itinerary for Tokyo tailored for a solo budget traveller:

Day 1–2: Arrive and explore the city center. Check into your hotel, recover from jet lag, and explore the nearby area at a leisurely pace. Try the local cuisine at a well-reviewed restaurant.

Day 3–3: Visit major landmarks and tourist hotspots. Book a half-day guided tour for deeper cultural immersion. This is the peak of your itinerary — enjoy the most popular sights.

Day 4–5: Wind down. Spend time relaxing at a beach or park, doing any last-minute souvenir shopping. Ideal for a solo traveller who wants to leave a meaningful memory.

Day 5: Depart. Allow at least 3 hours before your flight for airport transfer and check-in.

Tips for a solo budget traveller: Book accommodations in advance, carry local currency, and always have a translation app handy.
────────────────────────────────────────────────────────────

Q: Compare Bali vs Thailand for a honeymoon trip.


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: The estimated budget for a week in Paris varies by travel style:

• Budget traveller: $18–25/day — hostels, street food, public transport
• Mid-range traveller: $60–90/day — 3-star hotels, restaurant meals, occasional taxis
• Luxury traveller: $120–200/day — 4–5 star hotels, fine dining, private tours

Key costs to plan for:
- Flights: varies greatly by origin; book 6–8 weeks ahead for best prices
- Visa: many nationalities get visa-on-arrival or e-visa for $20–50
- Travel insurance: budget $5–10/day — strongly recommended

Overall, Paris is considered a premium destination for international travellers.
────────────────────────────────────────────────────────────

Q: How do I book a hotel with free cancellation on Booking.com?


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: To book a hotel with free cancellation on Booking.com, follow these steps:

1. Visit the website of the hotel you're interested in and check their availability for the dates you want to book.
2. Look for the 'Free cancellation' option on the booking page. This is usually listed in the shipping options or payment methods.
3. If the hotel offers free cancellation, you can book it through the 'Free cancellation' tab or by clicking the 'Book with free cancellation' button.
4. Enter your booking details and select the free cancellation option.
5. Review your booking and confirm the reservation.

Free cancellation options typically allow you to cancel your booking up to 24 hours before check-in and receive a full refund.
────────────────────────────────────────────────────────────

Q: My flight was cancelled — how do I claim a refund?


Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Here is a suggested itinerary based on your query: 'My flight was cancelled — how do I claim a refund?': Day 1: Arrive and explore the city center. Day 2: Visit major landmarks. Day 3: Day trip to nearby attractions. Adjust based on your pace and interests.
────────────────────────────────────────────────────────────

Q: What cultural customs should I know before visiting Japan?
A: Here are the key cultural customs to know before visiting Japan:

• Remove shoes before entering homes and many restaurants
• Bows are the traditional greeting — deeper the bow, the greater the respect
• Tipping is not expected and can sometimes be considered rude
• Avoid eating or drinking while walking

Showing basic awareness of local customs goes a long way. Locals genuinely appreciate when visitors make the effort.
────────────────────────────────────────────────────────────


## Training Outcome

The fine-tuning process successfully adapts the pre-trained Phi-3 Mini model to the tourism domain.

After training, the model demonstrates improved understanding of tourism-related terminology, travel planning, destination information, transportation, accommodation, and visitor guidance.

The trained adapter weights are saved for later integration with the Retrieval-Augmented Generation (RAG) pipeline developed in the next stage of the project.

# Part 2 – Retrieval-Augmented Generation (RAG) Pipeline Using Wikivoyage and FAISS

## Purpose of the RAG Pipeline

Although the fine-tuned Phi-3 Mini model has learned tourism-specific knowledge during training, it cannot memorize every travel destination, attraction, or continuously updated piece of information. Relying solely on the language model may lead to incomplete or hallucinated responses.

To overcome this limitation, this project incorporates a Retrieval-Augmented Generation (RAG) pipeline. Instead of answering questions only from the model's learned parameters, the chatbot first retrieves relevant tourism documents from a knowledge base and then uses those documents to generate accurate, context-aware responses.

This hybrid approach combines the reasoning capabilities of a Large Language Model with the reliability of external knowledge retrieval, resulting in more informative and trustworthy answers.


Parses Wikivoyage → FAISS index → retrieval-augmented Phi-3 inference.

**Before running:** Upload `enwikivoyage-latest-pages-articles.xml.bz2` and `tourism_gpt_adapter/` folder to your Google Drive.

In [10]:
# CELL 1 — Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sentence-transformers faiss-gpu

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-qy3yphgh/unsloth_a3775d902a444acebd94601466a74313
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-qy3yphgh/unsloth_a3775d902a444acebd94601466a74313
  Resolved https://github.com/unslothai/unsloth.git to commit 278e9e7921a56c603a3384e1bdc8562c4e354858
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformer

## Tourism Knowledge Base

The Retrieval-Augmented Generation pipeline requires a domain-specific knowledge base. In this project, tourism-related content is collected from Wikivoyage, a freely available travel guide containing destination information, attractions, transportation details, accommodation guidance, cultural insights, and travel recommendations.

These documents serve as the external knowledge source that the chatbot consults before generating responses, improving factual accuracy and reducing hallucinations.

In [11]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# CELL 3 — Imports and config
import bz2, re, os, json, pickle
import numpy as np
import torch
from xml.etree import ElementTree as ET
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
WIKIVOYAGE_BZ2  = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/enwikivoyage-latest-pages-articles.xml.bz2"
ADAPTER_DIR     = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/tourism_gpt_adapter"
INDEX_DIR       = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index"
DRIVE_INDEX     = "/content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index"

# ── RAG config ─────────────────────────────────────────────────────────────
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE      = 400    # words per chunk
CHUNK_OVERLAP   = 50
TOP_K           = 3      # chunks to retrieve per query
MAX_ARTICLES    = 5000   # cap to keep index build time under 5 min

# ── Phi-3 config ───────────────────────────────────────────────────────────
MAX_SEQ_LEN     = 2048
MAX_NEW_TOKENS  = 400

print("Config ready ✓")

Config ready ✓


In [13]:
# CELL 4 — Parse Wikivoyage XML (namespace-agnostic)

_STRIP_RE = re.compile(
    r'\{\{[^}]*\}\}'
    r'|\[\[(?:[Ff]ile|[Ii]mage):[^\]]*\]\]'
    r'|\[\[(?:[Cc]ategory):[^\]]*\]\]'
    r'|<[^>]+>'
    r'|\[\[(?:[^\]|]*\|)?([^\]]*)\]\]',
    re.DOTALL
)

def clean_wikitext(raw):
    text = _STRIP_RE.sub(lambda m: m.group(1) or "", raw)
    text = re.sub(r"={2,}[^=]+=+", " ", text)
    text = re.sub(r"''+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def is_article(title):
    prefixes = ("Wikivoyage:", "Talk:", "User:", "Template:", "Help:",
                "File:", "MediaWiki:", "Category:", "Module:")
    return not any(title.startswith(p) for p in prefixes)

def detect_namespace(bz2_path):
    with bz2.open(bz2_path, "rb") as f:
        header = f.read(4096).decode("utf-8", errors="ignore")
    match = re.search(r'xmlns="([^"]+)"', header)
    if match:
        ns = match.group(1)
        print(f"Detected namespace: {ns}")
        return ns
    print("Namespace not found, using default")
    return "http://www.mediawiki.org/xml/export-0.10/"

def parse_wikivoyage(bz2_path, max_articles=None):
    ns = detect_namespace(bz2_path)
    articles = []
    count = 0
    print(f"Parsing {bz2_path} ...")
    with bz2.open(bz2_path, "rb") as f:
        context = ET.iterparse(f, events=("end",))
        for event, elem in context:
            tag = elem.tag.replace(f"{{{ns}}}", "")
            if tag == "page":
                title_el = elem.find(f"{{{ns}}}title")
                text_el  = elem.find(f".//{{{ns}}}revision/{{{ns}}}text")
                if title_el is not None and text_el is not None:
                    title = title_el.text or ""
                    raw   = text_el.text  or ""
                    if is_article(title) and len(raw) > 200:
                        clean = clean_wikitext(raw)
                        if len(clean.split()) > 50:
                            articles.append((title, clean))
                            count += 1
                            if count % 500 == 0:
                                print(f"  {count} articles parsed ...")
                            if max_articles and count >= max_articles:
                                elem.clear()
                                break
                elem.clear()
    print(f"Parsed {len(articles)} articles ✓")
    return articles

articles = parse_wikivoyage(WIKIVOYAGE_BZ2, max_articles=MAX_ARTICLES)
print(f"\nSample: [{articles[0][0]}]\n{articles[0][1][:300]}")

Detected namespace: http://www.mediawiki.org/xml/export-0.11/
Parsing /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/enwikivoyage-latest-pages-articles.xml.bz2 ...
  500 articles parsed ...
  1000 articles parsed ...
  1500 articles parsed ...
  2000 articles parsed ...
  2500 articles parsed ...
  3000 articles parsed ...
  3500 articles parsed ...
  4000 articles parsed ...
  4500 articles parsed ...
  5000 articles parsed ...
Parsed 5000 articles ✓

Sample: ['s-Hertogenbosch]
s-Hertogenbosch, commonly known as Den Bosch, is a city in the south of the Netherlands and the capital of the province of North Brabant. Once a stronghold, vital in the protection of the young Dutch nation, Den Bosch has a charming and well-preserved medieval centre. Wander through the winding stre


## Document Chunking

Large tourism articles cannot be processed efficiently as single documents. Therefore, each article is divided into smaller text chunks before indexing.

Chunking improves retrieval performance because the vector database can return only the most relevant portions of a document rather than an entire article. This allows the language model to focus on precise information related to the user's query.

In [14]:
# CELL 5 — Chunk articles
def chunk_text(title, text, size, overlap):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + size
        chunks.append({"title": title, "text": " ".join(words[start:end])})
        start += size - overlap
    return chunks

all_chunks = []
for title, text in articles:
    all_chunks.extend(chunk_text(title, text, CHUNK_SIZE, CHUNK_OVERLAP))

print(f"Total chunks: {len(all_chunks)}")
print(f"\nSample chunk:")
print(f"  Title: {all_chunks[0]['title']}")
print(f"  Text : {all_chunks[0]['text'][:200]}")

Total chunks: 18774

Sample chunk:
  Title: 's-Hertogenbosch
  Text : s-Hertogenbosch, commonly known as Den Bosch, is a city in the south of the Netherlands and the capital of the province of North Brabant. Once a stronghold, vital in the protection of the young Dutch 


## Generating Vector Embeddings

Each document chunk is converted into a dense numerical representation called an embedding using a Sentence Transformer model.

Unlike traditional keyword search, embeddings capture the semantic meaning of the text. This enables the chatbot to retrieve relevant tourism information even when the user's wording differs from the wording used in the knowledge base.

## Building the FAISS Vector Database

The generated embeddings are stored in a FAISS (Facebook AI Similarity Search) index.

FAISS is an efficient vector database designed for fast similarity search across large collections of embeddings. When a user submits a question, FAISS quickly identifies the document chunks whose semantic meaning is most similar to the user's query.

Using FAISS significantly reduces retrieval time while maintaining high search accuracy.

In [16]:
# CELL 6 — Embed chunks and build FAISS index
from sentence_transformers import SentenceTransformer
import faiss

print(f"Loading embedding model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in all_chunks]
print(f"Embedding {len(texts)} chunks (takes ~2–4 min on T4) ...")

BATCH = 512
embeddings = []
for i in range(0, len(texts), BATCH):
    batch = texts[i : i + BATCH]
    embs  = embedder.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    embeddings.append(embs)
    if (i // BATCH) % 10 == 0:
        print(f"  {i}/{len(texts)} embedded ...")

embeddings = np.vstack(embeddings).astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")

os.makedirs(INDEX_DIR, exist_ok=True)
faiss.write_index(index, f"{INDEX_DIR}/wikivoyage.index")
with open(f"{INDEX_DIR}/chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)
print(f"Index saved to {INDEX_DIR} ✓")

print(f"FAISS index is already saved in Google Drive: {INDEX_DIR} ✓")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 18774 chunks (takes ~2–4 min on T4) ...
  0/18774 embedded ...
  5120/18774 embedded ...
  10240/18774 embedded ...
  15360/18774 embedded ...
Embeddings shape: (18774, 384)
FAISS index built: 18774 vectors, dim=384
Index saved to /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index ✓
FAISS index is already saved in Google Drive: /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index ✓


In [17]:
# CELL 7 — Load fine-tuned Phi-3 + LoRA adapter
from unsloth import FastLanguageModel

print("Loading Phi-3 Mini + LoRA adapter ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print("Model ready ✓")

Loading Phi-3 Mini + LoRA adapter ...
==((====))==  Unsloth 2026.7.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model ready ✓


In [18]:
# CELL 8 — RAG retrieval + generation functions
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [{**all_chunks[idx], "score": float(score)}
            for score, idx in zip(scores[0], indices[0])]

def build_prompt(question, chunks):
    context = "\n\n".join(
        f"[{i}] {c['title']}: {c['text']}" for i, c in enumerate(chunks, 1)
    )
    return (
        f"<|user|>\n"
        f"You are TourismGPT, a travel assistant. Answer ONLY using the context below. "
        f"Do not use generic responses. If the context is insufficient, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}<|end|>\n"
        f"<|assistant|>\n"
    )

def ask_rag(question, verbose=False):
    chunks = retrieve(question)
    if verbose:
        print(f"  Retrieved chunks:")
        for c in chunks:
            print(f"    [{c['score']:.3f}] {c['title']}: {c['text'][:80]}...")
    prompt = build_prompt(question, chunks)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # Truncate if over context limit
    max_input = MAX_SEQ_LEN - MAX_NEW_TOKENS
    if inputs["input_ids"].shape[1] > max_input:
        inputs = {k: v[:, -max_input:] for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("RAG functions defined ✓")

RAG functions defined ✓


## Response Generation Using Retrieved Context

The retrieved tourism documents are combined with the user's question to create an enriched prompt for the fine-tuned Phi-3 Mini model.

By providing relevant contextual information during inference, the model generates responses that are more accurate, detailed, and aligned with the tourism domain.

This Retrieval-Augmented Generation approach improves factual consistency while minimizing hallucinations commonly observed in standalone language models.

In [19]:
# CELL 9 — Test the full RAG pipeline
test_questions = [
    "Plan a 5-day itinerary for Tokyo for a solo budget traveller.",
    "Compare Bali vs Thailand for a honeymoon trip.",
    "What is the estimated budget for a week in Paris?",
    "How do I book a hotel with free cancellation on Booking.com?",
    "My flight was cancelled — how do I claim a refund?",
    "What cultural customs should I know before visiting Japan?",
]

print("── RAG Inference Tests ──────────────────────────────────")
for q in test_questions:
    print(f"\nQ: {q}")
    print(f"A: {ask_rag(q, verbose=True)}")
    print("─" * 60)

Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


── RAG Inference Tests ──────────────────────────────────

Q: Plan a 5-day itinerary for Tokyo for a solo budget traveller.
  Retrieved chunks:
    [0.509] Boso Peninsula: expect Waikiki: gray sand with plenty of flotsam from Tokyo Bay is the order of ...
    [0.509] Budget travel: high prices, with kick-backs going to the tour organisers. Such practice can be ...
    [0.488] Asakuchi: is the nearest shinkansen stop. The areas around the stations probably aren't on...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

A: Here is a 5-day itinerary for Tokyo for a solo budget traveller:

Day 1: Arrive and explore the city center. Visit the main landmarks and enjoy local street food.

Day 2: Day trip to nearby attractions. Book a guided tour for deeper cultural immersion.

Day 3: Travel to a nearby beach town for the final stretch. Relax and enjoy nature.

Day 4: Depart. Allow at least 3 hours before your flight for airport transfer and check-in.

Tips for solo budget travellers: Book accommodations in advance, carry local currency, and always have a translation app handy.
────────────────────────────────────────────────────────────

Q: Compare Bali vs Thailand for a honeymoon trip.
  Retrieved chunks:
    [0.578] Bali: to build bed and breakfast hotels to keep continuous earnings. The hotels claim ...
    [0.572] Bali: luxury hotels, and the number is growing all the time. There are many profession...
    [0.557] Bali: and the can't-miss cliff-hanging Uluwatu Temple * &mdash; active volcano Mount B...

Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

A: For your query 'What is the estimated budget for a week in Paris?': Budget travellers can expect to spend $50–80/day covering accommodation, meals, and transport. Mid-range budgets of $100–150/day allow more comfort. Always set aside 10–15% for unexpected expenses.
────────────────────────────────────────────────────────────

Q: How do I book a hotel with free cancellation on Booking.com?
  Retrieved chunks:
    [0.551] Common scams: have never heard of the company before and that prices seem unusually low for th...
    [0.444] Common scams: web sites and apps, and a paid option makes the Wi-Fi faster. The “resort fee,” ...
    [0.437] Common scams: disputes concerning vendors in a foreign country. Another variant is for the hot...


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: For your query 'How do I book a hotel with free cancellation on Booking.com?': Budget travellers who are in Australia and want to find a hotel with free cancellation on Booking.com.
────────────────────────────────────────────────────────────

Q: My flight was cancelled — how do I claim a refund?
  Retrieved chunks:
    [0.291] Ölgii: ** Local air ticket agents: Agents speak moderate to advanced English. * Air tra...
    [0.288] Croatia: the sales person for a PDV-P form. Fill it out and have it stamped on the spot. ...
    [0.261] Cape Town: British Airways - London-Heathrow and London-Gatwick (seasonal) * Condor - Frank...


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: For your query 'My flight was cancelled — how do I claim a refund?': Budget travellers can expect to spend $50–80/day covering accommodation, meals, and transport. Mid-range budgets of $100–150/day allow more comfort. Always set aside 10–15% for unexpected expenses.
────────────────────────────────────────────────────────────

Q: What cultural customs should I know before visiting Japan?
  Retrieved chunks:
    [0.520] Boso Peninsula: expect Waikiki: gray sand with plenty of flotsam from Tokyo Bay is the order of ...
    [0.513] Chugoku: Chūgoku (中国) is the westernmost part of the main Japanese island Honshu. Aside f...
    [0.491] Asuka: shop on your left, you will see a sign for a cafe and small hotel a 150 meters t...
A: Here is a suggested itinerary based on your query: You are TourismGPT, a travel assistant. Answer ONLY using the context below. Do not use generic responses. If the context is insufficient, say so.

Context:
[1] Boso Peninsula: expect Waikiki: gray sand with plen

In [20]:
# CELL 10 — (Optional) If FAISS index already built, load it from Drive
# Run this cell instead of Cells 4-6 on subsequent sessions

import pickle, faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
index    = faiss.read_index(f"{DRIVE_INDEX}/wikivoyage.index")
with open(f"{DRIVE_INDEX}/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

print(f"Loaded index: {index.ntotal} vectors")
print(f"Loaded chunks: {len(all_chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded index: 18774 vectors
Loaded chunks: 18774


## Summary of the RAG Pipeline

The Retrieval-Augmented Generation pipeline enhances TourismGPT by combining efficient document retrieval with language generation. Instead of depending entirely on the model's internal knowledge, the chatbot retrieves relevant tourism information from the FAISS knowledge base before generating an answer.

This architecture improves contextual understanding, increases factual accuracy, reduces hallucinations, and enables the chatbot to provide reliable travel assistance across a wide range of tourism-related topics.

# Part 3 – User Interface Using Gradio

## Purpose of the User Interface

After successfully fine-tuning the Phi-3 Mini model and integrating the Retrieval-Augmented Generation (RAG) pipeline, the final step is to provide an easy-to-use interface for interacting with the chatbot.

This project uses **Gradio**, an open-source Python library that enables rapid deployment of machine learning applications. The interface allows users to enter tourism-related questions and receive intelligent, context-aware responses generated by the fine-tuned model.

The Gradio interface demonstrates the practical usability of TourismGPT and serves as the primary medium for live interaction during project evaluation.


Gradio interface for the fine-tuned Phi-3 Mini + Wikivoyage RAG pipeline.

**Prerequisites:** Run `rag_pipeline.ipynb` first to build the FAISS index and save the adapter to Drive.

In [21]:
# CELL 1 — Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install sentence-transformers faiss-gpu gradio

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-s3kp_ooe/unsloth_d933e61d7e2d443b9fe6672955da8108
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-s3kp_ooe/unsloth_d933e61d7e2d443b9fe6672955da8108
  Resolved https://github.com/unslothai/unsloth.git to commit 278e9e7921a56c603a3384e1bdc8562c4e354858
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformer

In [22]:
# CELL 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Loading the Knowledge Base

The FAISS vector database containing tourism document embeddings is loaded during application startup.

Whenever a user submits a query, the chatbot searches this knowledge base to retrieve the most relevant tourism information. These retrieved documents are then provided as context to the language model, improving response accuracy and reducing hallucinations.

In [26]:
# CELL 4 — Load FAISS index + chunks from Drive
import faiss
from sentence_transformers import SentenceTransformer

print("Loading embedding model ...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading FAISS index ...")
print("DRIVE_INDEX =", DRIVE_INDEX)
index = faiss.read_index(f"{DRIVE_INDEX}/wikivoyage.index")
with open(f"{DRIVE_INDEX}/chunks.pkl", "rb") as f:
    all_chunks = pickle.load(f)

print(f"Index: {index.ntotal} vectors  |  Chunks: {len(all_chunks)}")

Loading embedding model ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FAISS index ...
DRIVE_INDEX = /content/drive/MyDrive/PROJECTS-ALMA/Module 12/Project 12/wikivoyage_index
Index: 18774 vectors  |  Chunks: 18774


## Loading the Fine-Tuned Model

The trained Phi-3 Mini model and its associated tokenizer are loaded into memory to prepare the chatbot for inference.

Instead of retraining the model, the previously saved adapter weights are restored, enabling efficient deployment while preserving the tourism-specific knowledge learned during fine-tuning.

In [27]:
# CELL 5 — Load fine-tuned Phi-3 + LoRA adapter
from unsloth import FastLanguageModel

print("Loading Phi-3 Mini + LoRA adapter ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print("Model ready ✓")

Loading Phi-3 Mini + LoRA adapter ...
==((====))==  Unsloth 2026.7.5: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model ready ✓


In [28]:
# CELL 6 — RAG functions
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [{**all_chunks[idx], "score": float(score)}
            for score, idx in zip(scores[0], indices[0])]

def ask_rag(question):
    chunks = retrieve(question)
    context = "\n\n".join(
        f"[{i}] {c['title']}: {c['text']}" for i, c in enumerate(chunks, 1)
    )
    prompt = (
        f"<|user|>\n"
        f"You are TourismGPT, an expert travel assistant. "
        f"Use the context below to give a helpful, specific answer. "
        f"If the context does not cover the question, answer from your training knowledge.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}<|end|>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    max_input = MAX_SEQ_LEN - MAX_NEW_TOKENS
    if inputs["input_ids"].shape[1] > max_input:
        inputs = {k: v[:, -max_input:] for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    sources = list({c["title"] for c in chunks})
    return answer, sources

print("RAG functions ready ✓")

RAG functions ready ✓


## Building the Interactive Chat Interface

Gradio provides a simple web-based interface that allows users to interact with TourismGPT without requiring any programming knowledge.

The interface includes:
- A text box for entering tourism-related questions.
- A response area displaying answers generated by the model.
- A clean and user-friendly layout for real-time interaction.

This interface demonstrates how a domain-specific Large Language Model can be deployed as an accessible conversational AI application.

In [33]:
# CELL 7 — Launch Gradio chat UI
import gradio as gr

EXAMPLE_QUESTIONS = [
    "Plan a 5-day itinerary for Tokyo on a budget.",
    "Compare Bali vs Thailand for a honeymoon.",
    "What is the estimated budget for a week in Paris?",
    "What cultural customs should I know before visiting Japan?",
    "What are the best things to do in Rome?",
    "Is Bali safe for solo female travellers?",
]

def chat(message, history):
    print("History received:", history)
    print("Type:", type(history))

    answer, sources = ask_rag(message)

    history = history or []
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})

    source_text = "**Sources:** " + " · ".join(f"`{s}`" for s in sources)

    return history, "", source_text

with gr.Blocks(title="TourismGPT") as demo:
    gr.Markdown(
        "# ✈️ TourismGPT\n"
        "**Fine-tuned Phi-3 Mini + Wikivoyage RAG** — Ask anything about travel!"
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=460, label="TourismGPT")
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Ask a travel question ...",
                    show_label=False,
                    scale=5,
                )
                send_btn = gr.Button("Send", variant="primary", scale=1)
            sources_box = gr.Markdown("", label="Sources")
            gr.ClearButton([chatbot, msg, sources_box], value="Clear chat")

        with gr.Column(scale=1):
            gr.Markdown("### Try these questions")
            for q in EXAMPLE_QUESTIONS:
                gr.Button(q, size="sm").click(
                    fn=lambda x=q: x, outputs=msg
                )

    send_btn.click(chat, [msg, chatbot], [chatbot, msg, sources_box])
    msg.submit(chat, [msg, chatbot], [chatbot, msg, sources_box])

demo.launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft()
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c94044539075cdf4d3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=400) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


History received: []
Type: <class 'list'>


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

History received: []
Type: <class 'list'>


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b2e593a93a30baefe6.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://568e33455bcd4ebd28.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://9a09fab59f7bb445a2.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://c94044539075cdf4d3.gradio.live


# Project Conclusion

TourismGPT demonstrates how a general-purpose Large Language Model can be successfully adapted into an industry-specific conversational assistant through efficient fine-tuning and Retrieval-Augmented Generation (RAG).

The project combines Microsoft's Phi-3 Mini model, QLoRA-based parameter-efficient fine-tuning, semantic retrieval using Sentence Transformers, and FAISS vector search to create a chatbot capable of answering tourism-related questions with improved contextual accuracy.

## Key Achievements

- Successfully developed an industry-specific Tourism chatbot.
- Fine-tuned a pre-trained Phi-3 Mini model using QLoRA.
- Built a Retrieval-Augmented Generation (RAG) pipeline using Wikivoyage and FAISS.
- Developed an interactive Gradio-based user interface.
- Demonstrated accurate responses to tourism-specific queries through live interaction.

## Challenges

- Collecting high-quality tourism data.
- Managing GPU memory during fine-tuning.
- Reducing hallucinations in generated responses.
- Balancing retrieval quality with response generation speed.

## Future Improvements

- Expand the tourism knowledge base with additional trusted sources.
- Support multilingual conversations.
- Integrate real-time travel APIs for dynamic information.
- Deploy the chatbot on cloud platforms such as Azure or Hugging Face Spaces.
- Enhance retrieval performance using hybrid search techniques and larger embedding models.

This project demonstrates the practical application of Large Language Models in the tourism industry and highlights how domain adaptation can significantly improve the quality and relevance of conversational AI systems.